# Task: Goldfish-to-shark adversarial attack

You are given a set of images that are classified by a model trained on Imagenet as a goldfish (class `GOLDFISH_LABEL=1`).<br>
Your task is to compute a strip of pixels of height 10, which, when pasted into the top 10 rows of *each* of the original images, makes the model classify each image as a shark (class `SHARK_LABEL=2`) with returned probability at least 0.5.

This should run under 2 minutes on CPU.

Chcemy zaatakować sieć

Założenia:
- Ustawione parametry sieci
- chcemy zmylic sieć żeby nie rozróżniała ryby i rekina

Chcemy znaleźć taki pasek w zdjęciach, żeby wyrzucała 

Żeby znaleźć taki element (pasek pikseli) trzeba policzyć trzeba, zrobić coś podobnego do trenowania sieci:
- trenujemy pasek, tak żeby otrzymywać odpowiedź shark, więc dopasowujemy model który ma przewidywać 
- loss po wejściach i optymalizujemy sieć 
- transformacja - renderowanie w sposób różniczkowalny 
    - rzut na przestrzeń 
    - ...


## Data

Below you can find a piece of code that download a zipfile and displays the images.

In [ ]:
import PIL.Image
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import zipfile
from io import BytesIO
from PIL import Image
from torch import Tensor
from torchvision.transforms import v2
from typing import cast

from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD


torch.use_deterministic_algorithms(True)


def get_images(
    url: str = "https://www.mimuw.edu.pl/~cygan/goldfish.zip",
    target_height: int = 100,
    target_width: int = 100,
) -> list[Tensor]:
    """Return five images as tensors of shape (C=3, H=target_height, W=target_width), 0..1 values."""
    request = requests.get(url)
    images = []
    with zipfile.ZipFile(BytesIO(request.content)) as z:
        for filename in z.namelist():
            if not filename.lower().endswith((".jpg", ".jpeg")):
                continue
            with z.open(filename) as f:
                images.append(Image.open(f).convert("RGB"))

    transformations = v2.Compose([
        v2.Resize(size=(target_height, target_width)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),  # scale to 0..1
    ])
    return [cast(Tensor, transformations(img)) for img in images]


def tensor_to_pil(image: Tensor) -> PIL.Image.Image:
    """Convert image tensor of shape (C, H, W), values 0..1, to PIL Image."""
    C, H, W = image.shape
    image = image.detach().permute(1, 2, 0).clone()  # to (H, W, C)
    img = PIL.Image.fromarray((image.numpy() * 255).astype(np.uint8))
    return img


normalize_colors = v2.Normalize(
    mean=IMAGENET_DEFAULT_MEAN,
    std=IMAGENET_DEFAULT_STD
)

In [ ]:
images = get_images()
for img in images:
    print(img.shape)
    display(tensor_to_pil(img))

## Model

We will be using the `mobilenet_v2` model, trained to do classification on the Imagenet dataset.
For this task, the model is a black-box: you do not have to know the details of the model's architecture.
In particular the model's weights should not be changed.

In [ ]:
import torch.nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

DEFAULT_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

def get_model(device: str = DEFAULT_DEVICE) -> torch.nn.Module:
    """Load a pretrained MobileNetV2 model.

    Input: normalized (not 0..1) images of shape (B, C, H, W),
    Output: logits (not probabilities or logprobs) of shape (B, num_classes=1000,).
    """
    model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
    model.to(torch.device(device))
    model.eval()
    model.requires_grad_(False)
    return model

GOLDFISH_LABEL = MobileNet_V2_Weights.IMAGENET1K_V1.meta["categories"].index("goldfish")
SHARK_LABEL = MobileNet_V2_Weights.IMAGENET1K_V1.meta["categories"].index("great white shark")
GOLDFISH_LABEL, SHARK_LABEL

In [ ]:
model = get_model('cpu')
model

## Evaluation

Below is a piece of code for:
* pasting the strip and running model inference on a normalized image;
* evaluating how well a strip performs, for this attack task.

In [ ]:
STRIP_HEIGHT = 10

def evaluate_strip(model: torch.nn.Module, images: list[Tensor], strip: Tensor) -> None:
    """
    Print goldfish and shark probabilities for each image with the given strip pasted on top.

    Args:
    - model: the classification model.
    - images: list of tensors of shape (3, H, W), values 0..1
    - strip: tensor of shape (3, STRIP_HEIGHT, W), values 0..1.
    """
    C, H, W = images[0].shape
    assert strip.shape == (C, STRIP_HEIGHT, W)
    assert strip.min() >= 0 and strip.max() <= 1, f"Strip should be 0..1, got {strip.min()=}, {strip.max()=}."

    device = next(model.parameters()).device

    for img in images:
        img = img.clone().to(device)
        img[:, :strip.shape[1], :] = strip.to(device)
        img = normalize_colors(img)
        logits = model(img.unsqueeze(0)).squeeze(0)  # Shape (num_classes=1000,)
        probabilities = torch.nn.functional.softmax(logits, dim=-1)
        p_goldfish, p_shark = probabilities[[GOLDFISH_LABEL, SHARK_LABEL]]
        print(f'Probabilities assigned to: goldfish={p_goldfish:.5%}, shark={p_shark:.5%}.')


zero_strip = torch.zeros(3, STRIP_HEIGHT, images[0].shape[2])
evaluate_strip(model, images, zero_strip)

## TO IMPLEMENT

Here you have to implement the function that finds the strip.


In [ ]:
model(images[-1][None, ...]).shape
torch.arange(1, 20)

In [ ]:
import torch.nn as nn

def calculate_strip(model: torch.nn.Module, images: list[Tensor], seed: int = 1, quiet: bool = False) -> Tensor:
    device = next(model.parameters()).device
    torch.manual_seed(seed)

    # TODO uwagi - trzeba torchem myśleć.i jak naszybko 
    # do adama, można zacząć koło 1 i adam sam dobierze odpowiedni parametr w trakcie
    # to powoduje że nie trzeba szukać hiperparametrów
    # 

    ##{ TODO START
    param = ... #TODO tutaj trzeba wziąć obiekt który będziemy optymalizować (ta cecha)
    optimizer = torch.optim.Adam([param], lr=0.1)

    for epoch in range(300):
        # TODO ta pętla możę być nie potrzebna, ale tutaj model nie obsługiwał wielu inputów 
        for obj in objcs:
            # rozpakowanie obrazu
            optimizer.zero_grad()
            # TODO wczytanie danych i zrobienie podmiany 
            input_data = ...
            # liczymy nasz model
            logits = model(input_data) # trzeba to 
            # TODO odpowiednia transformacja (ta metryka)
            logits_transform = ...
            # TODO odpowiendia funkcja kosztu ktora będzie porównywać wyniki tych rzeczy
            loss = nn.functional.kl_div(logits_transform, reduction='batchmean')
            
            # aktualizacja wag
            loss.backward()
            optimizer.step()

        # sprawdzamy błą 
        if epoch % 10 == 0:
            print(f'Epoka {epoch+1}, strata: {loss.item():.4f}')
    ##} END TODO
    return zero_strip


your_strip = calculate_strip(model, images)

#TODO - zrozumienie struktury biblioteki - szybko to trzeba obsługiwać 
#TODO - czy tutaj nie powinno być jakoś podawanego tego stripa, chociaż nie trzeba tego robic 

In [ ]:
import torch.nn as nn

def calculate_strip(model: torch.nn.Module, images: list[Tensor], seed: int = 1, quiet: bool = False) -> Tensor:
    device = next(model.parameters()).device
    torch.manual_seed(seed)

    # TODO uwagi - trzeba torchem myśleć.i jak naszybko 
    # do adama, można zacząć koło 1 i adam sam dobierze odpowiedni parametr w trakcie
    # to powoduje że nie trzeba szukać hiperparametrów
    # 

    ##{ TODO START
    param = torch.rand_like(zero_strip, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([param], lr=0.1)
    target_class = [SHARK_LABEL, GOLDFISH_LABEL]
    target_class = [SHARK_LABEL]

    for epoch in range(300):
        for img in images:
            # rozpakowanie obrazu
            optimizer.zero_grad()
            # taking image (script copied from function above)
            img = img.clone().to(device)
            # normalizujemy nasz parametr - zapewniamy że jest w (0, 1)
            img[:, :param.shape[1], :] = torch.sigmoid(param)
            img = normalize_colors(img)
            # liczymy nasz model
            logits = model(img.unsqueeze(0)) # (1, 1000) 1000 klas
            log_probs = nn.functional. log_softmax(logits[:, target_class], dim=-1) # (1, 2)
            # print(log_probs)
            # kl_div samemu zamienia nasze 1., 0. na logarytm 
            loss = nn.functional.kl_div(log_probs, target=torch.tensor([[1.]]), reduction='batchmean')
            
            loss.backward()
            optimizer.step()

        if epoch % 10 == 0:
            print(f'Epoka {epoch+1}, strata: {loss.item():.4f}')
            evaluate_strip(model, images, torch.sigmoid(param))

    ##} END TODO
    return zero_strip


your_strip = calculate_strip(model, images)

#TODO - zrozumienie struktury biblioteki - szybko to trzeba obsługiwać 
#TODO - czy tutaj nie powinno być jakoś podawanego tego stripa, chociaż nie trzeba tego robic 

In [ ]:
import torch.nn as nn

def calculate_strip(model: torch.nn.Module, images: list[Tensor], seed: int = 1, quiet: bool = False) -> Tensor:
    device = next(model.parameters()).device
    torch.manual_seed(seed)

    # TODO uwagi - trzeba torchem myśleć.i jak naszybko 
    # do adama, można zacząć koło 1 i adam sam dobierze odpowiedni parametr w trakcie
    # to powoduje że nie trzeba szukać hiperparametrów
    # 

    ##{ TODO START
    param = torch.rand_like(zero_strip, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([param], lr=0.4)
    target = torch.tensor([SHARK_LABEL])

    for epoch in range(300):
        for img in images:
            # rozpakowanie obrazu
            optimizer.zero_grad()
            # taking image (script copied from function above)
            img = img.clone().to(device)
            # normalizujemy nasz parametr - zapewniamy że jest w (0, 1)
            img[:, :param.shape[1], :] = torch.sigmoid(param)
            img = normalize_colors(img)
            # liczymy nasz model
            logits = model(img.unsqueeze(0))
            loss = nn.functional.cross_entropy(logits, target)
            
            loss.backward()
            optimizer.step()

        if epoch % 10 == 0:
            print(f'Epoka {epoch+1}, strata: {loss.item():.4f}')
            evaluate_strip(model, images, torch.sigmoid(param))

    ##} END TODO
    return zero_strip


your_strip = calculate_strip(model, images)

#TODO - zrozumienie struktury biblioteki - szybko to trzeba obsługiwać 
#TODO - czy tutaj nie powinno być jakoś podawanego tego stripa, chociaż nie trzeba tego robic 

wersja z NLL 

In [ ]:
import torch.nn as nn

def calculate_strip(model: torch.nn.Module, images: list[Tensor], seed: int = 1, quiet: bool = False) -> Tensor:
    device = next(model.parameters()).device
    torch.manual_seed(seed)

    # TODO uwagi - trzeba torchem myśleć.i jak naszybko 
    # do adama, można zacząć koło 1 i adam sam dobierze odpowiedni parametr w trakcie
    # to powoduje że nie trzeba szukać hiperparametrów
    # 

    ##{ TODO START
    param = torch.rand_like(zero_strip, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([param], lr=0.4)
    target = torch.tensor([SHARK_LABEL])

    for epoch in range(300):
        for img in images:
            # rozpakowanie obrazu
            optimizer.zero_grad()
            # taking image (script copied from function above)
            img = img.clone().to(device)
            # normalizujemy nasz parametr - zapewniamy że jest w (0, 1)
            img[:, :param.shape[1], :] = torch.sigmoid(param)
            img = normalize_colors(img)
            # liczymy nasz model
            logits = model(img.unsqueeze(0))
            loss = nn.functional.nll_loss(logits, target)
            
            loss.backward()
            optimizer.step()

        if epoch % 10 == 0:
            print(f'Epoka {epoch+1}, strata: {loss.item():.4f}')
            evaluate_strip(model, images, torch.sigmoid(param))

    ##} END TODO
    return zero_strip


your_strip = calculate_strip(model, images)

#TODO - zrozumienie struktury biblioteki - szybko to trzeba obsługiwać 
#TODO - czy tutaj nie powinno być jakoś podawanego tego stripa, chociaż nie trzeba tego robic 

Zaliczeniowe 

mamy dwie nakładki 

1. do klasyfikacji
2. do regresji 